# Thème Numéro 2 - Facteurs Saisonniers et Succès au Speed Dating

## Question 2 - La sélectivité varie-t-elle selon la saison ?
La sélectivité des participants lors du speed dating diffère-t-elle entre le printemps et l'automne ?

- **H0** : il n'y a pas de différence de sélectivité entre le printemps et l'automne (p > 0.05)
- **H1** : la sélectivité diffère selon la saison (p ≤ 0.05)

**Variables analysées :**
- `taux_dec` : taux de décisions positives par participant = `nb_oui / round`
  - On utilise `dec == 1` (décision **individuelle** du participant) plutôt que `match == 1` (réciprocité des deux personnes). Cela mesure directement la **sélectivité propre** du participant : à quelle fréquence dit-il oui, indépendamment de ce que l'autre personne décide.
  - `round` est identique pour tous les participants d'une même vague — on peut donc l'utiliser directement comme dénominateur sans recalcul individuel.
- `saison` : Spring (vagues 6–9 et 18-21) ou Autumn (vagues 1–5 et 10–17)

Seuil de significativité : α = 0.05

## 0. Chargement des données

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("Speed+Dating+Data.csv", encoding="MacRoman")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Columns: 195 entries, iid to amb5_3
dtypes: float64(174), int64(13), object(8)
memory usage: 12.5+ MB


## 1. Création des variables

### Mapping vague -> saison
Le dataset contient 21 vagues (*waves*) d'événements.
D'après la documentation, les vagues 6 à 9 et 18 à 21 se sont tenues au **printemps** (*Spring*), toutes les autres (1–5, 10–17) à l'**automne** (*Autumn*).

### Taux de décision positive (sélectivité)
On calcule pour chaque individu:
$$\text{taux\_dec}_i = \frac{\sum(\text{dec} == 1)}{\text{round}}$$

Ce taux est compris entre 0 (le participant n'a dit oui à personne) et 1 (il a dit oui à tout le monde).
Un taux **faible** indique une forte sélectivité (peu de oui), un taux **élevé** une faible sélectivité (beaucoup de oui).
Ce taux ne dépend que du comportement propre du participant - pas de la décision de l'autre personne (comme pour le match).

In [7]:
SPRING_WAVES = [6, 7, 8, 9, 18, 19, 20, 21]

df['season'] = df['wave'].apply(lambda w: 'Spring' if w in SPRING_WAVES else 'Autumn')

per_person = df.groupby(['iid', 'wave', 'season']).agg(
    nb_oui=('dec', 'sum'),
    round=('round', 'first')
).reset_index()

per_person['taux_dec'] = per_person['nb_oui'] / per_person['round']

print(f"Nombre de participants : {len(per_person)}")
print(per_person['season'].value_counts().to_string())
per_person.head()

Nombre de participants : 551
season
Autumn    350
Spring    201


,iid,wave,season,nb_oui,round,taux_dec
0,1,1,Autumn,8,10,0.8
1,2,1,Autumn,4,10,0.4
2,3,1,Autumn,0,10,0.0
3,4,1,Autumn,3,10,0.3
4,5,1,Autumn,6,10,0.6


## 2. Statistiques descriptives

In [9]:
descr = per_person.groupby('season')['taux_dec'].describe().round(2)
print("Statistiques descriptives du taux de match par saison :")
print(descr)

Statistiques descriptives du taux de match par saison :
        count  mean   std  min   25%   50%   75%  max
season                                               
Autumn  350.0  0.43  0.25  0.0  0.24  0.41  0.57  1.0
Spring  201.0  0.43  0.26  0.0  0.20  0.40  0.60  1.0


## 3. Vérification des conditions du t-test

Avant de réaliser un t-test indépendant, il faut vérifier la **normalité** (test de Shapiro-Wilk sur chaque groupe)


In [10]:
spring_data = per_person[per_person['season'] == 'Spring']['taux_dec']
autumn_data = per_person[per_person['season'] == 'Autumn']['taux_dec']

# normalité
stat_s, p_shapiro_s = stats.shapiro(spring_data)
stat_a, p_shapiro_a = stats.shapiro(autumn_data)

print("Test de Shapiro-Wilk (normalité):\n")
print(f"Spring : W = {stat_s:.3f}, p = {p_shapiro_s:.4f}")
print(f"Autumn : W = {stat_a:.3f}, p = {p_shapiro_a:.4f}")

if p_shapiro_s < 0.05 or p_shapiro_a < 0.05:
    print("\nAu moins un groupe ne suit pas une distribution normale (p < 0.05).")
else:
    print("\nLes deux groupes suivent une distribution normale.")


Test de Shapiro-Wilk (normalité):

Spring : W = 0.971, p = 0.0004
Autumn : W = 0.971, p = 0.0000

Au moins un groupe ne suit pas une distribution normale (p < 0.05).


## 4. T-test indépendant : Spring vs Autumn

On compare le taux de décision positive moyen entre les deux saisons via un **t-test indépendant**.
Les deux groupes sont indépendants (participants différents selon la vague).

In [11]:
t_stat, p_ttest = stats.ttest_ind(spring_data, autumn_data)

print("T-test indépendant:")
print(f"t = {t_stat:.3f}")
print(f"p-value = {p_ttest:.4f}")

if p_ttest < 0.05:
    print("\nH0 rejetée : différence significative de sélectivité entre les saisons (p ≤ 0.05)")
    if spring_data.mean() > autumn_data.mean():
        print("-> Les participants du printemps disent oui plus souvent (moins sélectifs).")
    else:
        print("-> Les participants de l'automne dient oui plus souvent (moins sélectifs).")
else:
    print("\nH0 non rejetée : pas de différence significative de sélectivité entre les saisons (p > 0.05)")

T-test indépendant:
t = -0.022
p-value = 0.9827

H0 non rejetée : pas de différence significative de sélectivité entre les saisons (p > 0.05)


## 5. Récapitulatif

In [12]:
print("Récapitulatif :")
print(f"  Taux de décision positive moyen (Spring) : {spring_data.mean():.4f}  (n = {len(spring_data)})")
print(f"  Taux de décision positive moyen (Autumn) : {autumn_data.mean():.4f}  (n = {len(autumn_data)})")
print(f"  Différence de taux                       : {abs(spring_data.mean() - autumn_data.mean()):.4f}")
print()
print(f"  T-test : t = {t_stat:.3f}, p = {p_ttest:.4f}")

Récapitulatif :
  Taux de décision positive moyen (Spring) : 0.4262  (n = 201)
  Taux de décision positive moyen (Autumn) : 0.4267  (n = 350)
  Différence de taux                       : 0.0005

  T-test : t = -0.022, p = 0.9827


## 6. Visualisation

Les graphiques ci-dessous illustrent la distribution du taux de décision positive par saison ainsi que la comparaison des moyennes.

In [15]:
# box plot : distribution du taux de décision positive par saison
fig_box = px.box(
    per_person,
    x='season',
    y='taux_dec',
    color='season',
    title="Distribution du taux de décision positive par saison (sélectivité)",
    labels={
        'taux_dec': 'Taux de décision positive (oui / rencontres)',
        'season': 'Saison'
    },
    color_discrete_map={'Spring': '#2ecc71', 'Autumn': '#e67e22'}
)
fig_box.show()

In [16]:
# histogramme : fréquence du taux de décision positive par saison
fig_hist = px.histogram(
    per_person,
    x='taux_dec',
    color='season',
    barmode='overlay',
    opacity=0.7,
    nbins=20,
    title="Distribution du taux de décision positive - Spring vs Autumn",
    labels={
        'taux_dec': 'Taux de décision positive (oui / rencontres)',
        'count': 'Nombre de participants',
        'season': 'Saison'
    },
    color_discrete_map={'Spring': '#2ecc71', 'Autumn': '#e67e22'}
)
fig_hist.show()

In [17]:
# bar chart : comparaison des taux de décision moyens avec erreur standard
summary = per_person.groupby('season')['taux_dec'].agg(['mean', 'std', 'count']).reset_index()
summary['sem'] = summary['std'] / np.sqrt(summary['count'])

fig_bar = px.bar(
    summary,
    x='season',
    y='mean',
    error_y='sem',
    color='season',
    text=summary['mean'].round(4),
    title="Taux de décision positive moyen par saison (± erreur standard)",
    labels={
        'mean': 'Taux de décision positive moyen',
        'season': 'Saison'
    },
    color_discrete_map={'Spring': '#2ecc71', 'Autumn': '#e67e22'}
)
fig_bar.update_traces(textposition='outside')
fig_bar.show()

## Conclusion

Le t-test indépendant ne permet pas de rejeter H0:
- **T-test indépendant** :t = -0.022, p = 0.983

**Il n'existe pas de différence significative de sélectivité entre le printemps et l'automne.**

Les taux de décision positive moyens sont quasiment identiques:
- **Spring**: 0.4262 (n = 201) - les participants disent oui à environ 43% de leurs rencontres.
- **Autumn**: 0.4267 (n = 350) - les participants disent oui à environ 43% de leurs rencontres.
- **Différence**: 0.0005, soit moins d'un millième de point.

Comme le confirment les visualisations, les distributions se superposent presque parfaitement: même médiane (~0.40–0.41), même dispersion interquartile (Q1 ≈ 0.20–0.24,
Q3 ≈ 0.57–0.60), et des barres de moyennes presque à la même hauteur.

Contrairement au taux de match de la première question,, cette variable mesure le comportement **individuel** du participant indépendamment de la décision de l'autre.
Le résultat reste néanmoins identique: la saison n'influence pas le comportement de sélection.

On note également que le taux de décision positive (~43%) est bien supérieur au taux de match (~17% dans la première question), ce qui confirme que les matchs sont contraints par la réciprocité - beaucoup de "oui" ne se transforment pas en match faute de réciprocité de l'autre côté.

### Ce que ça nous dit
La sélectivité individuelle des participants est **stable quelle que soit la saison**: ni le printemps ni l'automne ne poussent les participants à dire oui plus ou moins souvent.
Combiné aux résultats de la première question, on peut conclure que la saison n'est un facteur contextuel pertinent ni pour le succès ni pour le comportement de sélection au speed dating.

### Limites à considérer
- La normalité n'est pas vérifiée dans les deux groupes (Shapiro-Wilk p < 0.001), mais le t-test reste valide pour des échantillons de cette taille.
- Le groupe Spring reste le plus petit (n = 201 vs n = 350), ce qui limite la puissance statistique pour détecter un effet potentiellement faible.
- Le taux de décision positive ne distingue pas entre sélectivité réelle et comportement stratégique (certains participants peuvent dire oui à tout le monde pour maximiser leurs chances).